# 001-seg-map-correction — correlator walkthrough

# Imports

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.segmentation import CHANNEL_SETS, SegmentationFeatures, load_unet
from uavloc.features.vgg11 import Vgg11Features, load_vgg11
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.correlator import Correction, CorrelatorConfig, PositionCorrelator
from uavloc.plotting.correlation import (CORRECTED_COLOURS, SEG_PALETTE, class_colours, max_activation, plot_heatmap, plot_match,
                                         plot_overview, show_patch)
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "001-seg-map-correction"
RUNS = {"vgg11": "001.1", "segmentation": "001.2"}
NOISE_M = 15.0
DEVICE = "mps"

WALKTHROUGH_FRAME = None          # None -> the reachable draw where VGG11's error is closest to its median
WALKTHROUGH_BEARING_DEG = None
N_EXTREMES = 2                    # best and worst draws per method in the gallery
OVERVIEW_MARGIN_M = 85.0          # window corners reach ~80 m from the truth at 15 m

SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
runs = {method: next(r for r in load_runs(EXP) if r["run"] == run_id) for method, run_id in RUNS.items()}
config = runs["segmentation"]["config"]
frames = {f.name: f for f in load_frames(config["frames_dataset"])}
satellite = GeoImage.load(REPO_ROOT / config["sat_image"], LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))

In [ ]:
def load_records(run_id: str) -> pd.DataFrame:
    records = pd.read_csv(experiment_dir(EXP) / "artifacts" / f"records-{run_id}.csv")
    return records[records["noise_m"] == NOISE_M].set_index(["frame", "bearing_deg"])

In [ ]:
records = {method: load_records(run_id) for method, run_id in RUNS.items()}
draws = records["vgg11"][["prior_lat", "prior_lon", "reachable"]].copy()
for method, r in records.items():
    draws[f"error_{method}"] = r["error_m"]
draws = draws.reset_index()
draws.describe().loc[["mean", "50%"], ["error_vgg11", "error_segmentation"]]

# Model

In [ ]:
vgg11_features = Vgg11Features(load_vgg11(REPO_ROOT / config["vgg11_checkpoint"], DEVICE), satellite,
                               config["drone_input_px"], config["search_scale"], DEVICE)
seg_features = SegmentationFeatures(load_unet(REPO_ROOT / config["seg_checkpoint_drone"], DEVICE),
                                    load_unet(REPO_ROOT / config["seg_checkpoint_sat"], DEVICE), satellite,
                                    config["seg_channels"], config["drone_seg_gsd_m"], config["match_gsd_m"], DEVICE)
correlator_config = CorrelatorConfig(config["search_scale"], config["fov_w_deg"],
                                     sat_feature_crop_pct=config["sat_feature_crop_pct"],
                                     drone_feature_crop_pct=config["drone_feature_crop_pct"])

In [ ]:
captured = {}
correlators = {
    method: PositionCorrelator(features, correlator_config,
                               features_hook=lambda drone, sat, method=method: captured.__setitem__(method, (drone, sat)))
    for method, features in {"vgg11": vgg11_features, "segmentation": seg_features}.items()
}

# Satellite class map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
probabilities = seg_features.probabilities
axes[0].imshow(satellite.resampled(config["match_gsd_m"]).pixels)
axes[1].imshow(SEG_PALETTE[probabilities.pixels.argmax(-1)])
xs, ys = zip(*(probabilities.to_px(f.position) for f in frames.values()))
for ax in axes:
    ax.plot(xs, ys, "c.", ms=4); ax.set_xticks([]); ax.set_yticks([])
axes[1].set_title(f"satellite classes, {config['seg_checkpoint_sat'].rsplit('/', 1)[-1]}")
plt.tight_layout(); plt.show()

# Run

In [ ]:
def trace(frame: DroneFrame, prior: LatLon) -> dict[str, tuple[Correction, object, object]]:
    """Each method's correction plus the drone and satellite feature maps its hook saw."""
    traced = {}
    for method, correlator in correlators.items():
        correction = correlator.correct(frame, prior)
        traced[method] = (correction, *captured[method])
    return traced

In [ ]:
def pick_walkthrough() -> pd.Series:
    if WALKTHROUGH_FRAME is not None:
        match = draws[(draws["frame"] == WALKTHROUGH_FRAME) & np.isclose(draws["bearing_deg"], WALKTHROUGH_BEARING_DEG)]
        return match.iloc[0]
    reachable = draws[draws["reachable"]]
    return reachable.iloc[(reachable["error_vgg11"] - reachable["error_vgg11"].median()).abs().argmin()]

In [ ]:
draw = pick_walkthrough()
frame, prior = frames[draw["frame"]], LatLon(draw["prior_lat"], draw["prior_lon"])
traced = trace(frame, prior)
draw

In [ ]:
{method: {"error_m": haversine_m(c.position, frame.position), "logged_error_m": draw[f"error_{method}"], "peak_z": c.peak_z}
 for method, (c, _, _) in traced.items()}

In [ ]:
correlator = correlators["vgg11"]
patch_size_m = correlator.patch_size_m(frame)
footprint_m = correlator.footprint_m(frame)
patch_rgb = satellite.crop_rotated(prior, frame.yaw_deg, *patch_size_m)
true_offset_m = correlator.patch_offset_m(prior, frame.position, frame.yaw_deg)
patch_size_m, footprint_m, true_offset_m

# Step 1: prior, truth and search window

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_overview(ax, satellite, frame, prior, {m: c.position for m, (c, _, _) in traced.items()}, patch_size_m, OVERVIEW_MARGIN_M)
ax.set_title(f"{frame.name}  yaw {frame.yaw_deg:.0f}  alt {frame.altitude_m:.0f} m  bearing {draw['bearing_deg']:.0f}")
plt.show()

# Step 2: drone frame and heading-up satellite patch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1, patch_size_m[0] / footprint_m[0]]})
axes[0].imshow(frame.image()); axes[0].set_title("drone frame (top = heading)", fontsize=9); axes[0].axis("off")
show_patch(axes[1], patch_rgb, patch_size_m, "satellite patch around the prior, rotated to heading")
plt.tight_layout(); plt.show()

# Step 3: feature maps

In [ ]:
class_ids = list(CHANNEL_SETS[config["seg_channels"]])
views = {
    "vgg11": lambda fmap: max_activation(fmap),
    "segmentation": lambda fmap: class_colours(fmap, class_ids),
}
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for row, (method, (_, drone_map, sat_map)) in zip(axes, traced.items()):
    row[0].imshow(views[method](drone_map)); row[0].set_title(f"{method}: drone {tuple(drone_map.shape[1:])}", fontsize=9)
    row[1].imshow(views[method](sat_map)); row[1].set_title(f"{method}: satellite patch {tuple(sat_map.shape[1:])}", fontsize=9)
    for ax in row: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# Step 4: correlation heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (method, (correction, _, _)), colour in zip(axes, traced.items(), CORRECTED_COLOURS):
    shown = plot_heatmap(ax, patch_rgb, correction, true_offset_m, colour, f"{method}: peak z {correction.peak_z:.2f}")
    fig.colorbar(shown, ax=ax, shrink=.8, label="z-score")
axes[0].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

# Step 5: matched footprint vs true footprint

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (method, (correction, _, _)), colour in zip(axes, traced.items(), CORRECTED_COLOURS):
    error_m = haversine_m(correction.position, frame.position)
    plot_match(ax, patch_rgb, correction, true_offset_m, footprint_m, colour, f"{method}: error {error_m:.1f} m")
axes[0].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

# Gallery: best and worst draws

In [ ]:
def gallery_draws(method: str) -> pd.DataFrame:
    ranked = draws.sort_values(f"error_{method}")
    return pd.concat([ranked.head(N_EXTREMES), ranked.tail(N_EXTREMES)]).assign(picked_by=method)

In [ ]:
def plot_draw_row(axes: np.ndarray, row: pd.Series) -> None:
    frame, prior = frames[row["frame"]], LatLon(row["prior_lat"], row["prior_lon"])
    traced = trace(frame, prior)
    correlator = correlators["vgg11"]
    patch_rgb = satellite.crop_rotated(prior, frame.yaw_deg, *correlator.patch_size_m(frame))
    true_offset_m = correlator.patch_offset_m(prior, frame.position, frame.yaw_deg)
    axes[0].imshow(frame.image()); axes[0].axis("off")
    axes[0].set_title(f"{row['frame'][-7:]} bearing {row['bearing_deg']:.0f}\npicked: {row['picked_by']}", fontsize=8)
    for ax, (method, (correction, _, _)), colour in zip(axes[1:3], traced.items(), CORRECTED_COLOURS):
        plot_heatmap(ax, patch_rgb, correction, true_offset_m, colour,
                     f"{method}: {haversine_m(correction.position, frame.position):.1f} m, z {correction.peak_z:.1f}")
    plot_overview(axes[3], satellite, frame, prior, {m: c.position for m, (c, _, _) in traced.items()},
                  correlator.patch_size_m(frame), OVERVIEW_MARGIN_M)

In [ ]:
gallery = pd.concat([gallery_draws(method) for method in RUNS])
fig, axes = plt.subplots(len(gallery), 4, figsize=(22, 4.2 * len(gallery)))
for row_axes, (_, row) in zip(axes, gallery.iterrows()):
    plot_draw_row(row_axes, row)
plt.tight_layout(); plt.show()